In [1]:
import os

from openai import OpenAI

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

client = OpenAI(api_key=OPENAI_API_KEY)

completion = client.chat.completions.create(
    model='gpt-3.5-turbo-0125',
    messages=[{'role': 'user', 'content': 'hi'}],
    temperature=0.0,
)

In [2]:
import json

with open('./res/reviews.json', 'r') as f:
    review_list = json.load(f)

review_list[:3]

[{'title': '子供には最高屋上スパ',
  'review': '2家族10人で利用させて頂きました。和室湖畔側のお部屋は景色も良く清潔感あふれるお部屋でした。夕食朝食ともに大満足でした。なにより子供達が屋上スパに大大大満足でした。いつも忙しく子供との時間もとれて無かったので。とても良い旅行になりました。ありがとう感謝',
  'star': 5,
  'date': '2025年12月',
  'room': 5,
  'bath': 5,
  'breakfast': 5,
  'dinner': 5,
  'service': 5,
  'cleanliness': 5},
 {'title': '大変満足してます',
  'review': 'ご飯も美味しくて、温泉もよく、部屋もそれなりでよかった。花火も綺麗にみることができて、満足しています。また利用したいと思います。',
  'star': 4,
  'date': '2026年2月',
  'room': 4,
  'bath': 4,
  'breakfast': 4,
  'dinner': 4,
  'service': 4,
  'cleanliness': 4},
 {'title': '夜明けの天空スパはステキ',
  'review': '16時前にチェックインしたのですが既に18時台のお食事は満席でした。湖が見える良い部屋だったため20時の花火はお部屋で見たかったので、19時のお食事だとせっかくのバイキングも何となく急かされて、途中でお部屋に戻ってからまた食事席に着くのもなぁと…少し残念でした。(まだお食事中って札を置いて席を立つことは可能)夕食も朝食も種類が沢山あって、選ぶのが楽しかったです。天空スパは早朝に入りましたが、段々夜が明けていく感じがステキでした。',
  'star': 5,
  'date': '2026年2月',
  'room': 5,
  'bath': 5,
  'breakfast': 5,
  'dinner': 5,
  'service': 5,
  'cleanliness': 5}]

In [3]:
good_cnt, bad_cnt = 0, 0
for r in review_list:
    if r['star'] == 5:
        good_cnt += 1
    else:
        bad_cnt += 1

print(f'高評価（5点）: {good_cnt}件, それ以外: {bad_cnt}件')

高評価（5点）: 53件, それ以外: 97件


In [4]:
reviews_good, reviews_bad = [], []

for r in review_list:
    if r['star'] == 5:
        reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
    else:
        reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

reviews_good_text = '\n'.join(reviews_good)
reviews_bad_text = '\n'.join(reviews_bad)

In [5]:
import datetime
import re

def preprocess_reviews(path='./res/reviews.json'):
    with open(path, 'r', encoding='utf-8') as f:
        review_list = json.load(f)

    reviews_good, reviews_bad = [], []

    current_date = datetime.datetime.now()
    date_boundary = current_date - datetime.timedelta(days=6*30)

    for r in review_list:
        review_date_str = r.get('date', '')
        # 「YYYY年MM月」形式の日付をパース
        m = re.match(r'(\d{4})年(\d{1,2})月', review_date_str)
        if m:
            review_date = datetime.datetime(int(m.group(1)), int(m.group(2)), 1)
        else:
            review_date = current_date

        if review_date < date_boundary:
            continue

        if r['star'] == 5:
            reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
        else:
            reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

    reviews_good_text = '\n'.join(reviews_good)
    reviews_bad_text = '\n'.join(reviews_bad)

    return reviews_good_text, reviews_bad_text

good, bad = preprocess_reviews()

In [ ]:
def pairwise_eval(reviews, answer_a, answer_b):
    eval_prompt = f"""[System]
Please act as an impartial judge and evaluate the quality of the Japanese summaries provided by two
AI assistants to the set of user reviews on accommodations displayed below. You should choose the assistant that
follows the user's instructions and answers the user's question better. Your evaluation
should consider factors such as the helpfulness, relevance, accuracy, depth, creativity,
and level of detail of their responses. Begin your evaluation by comparing the two
responses and provide a short explanation. Avoid any position biases and ensure that the
order in which the responses were presented does not influence your decision. Do not allow
the length of the responses to influence your evaluation. Do not favor certain names of
the assistants. Be as objective as possible. After providing your explanation, output your
final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]"
if assistant B is better, and "[[C]]" for a tie.
[User Reviews]
{reviews}
[The Start of Assistant A's Answer]
{answer_a}
[The End of Assistant A's Answer]
[The Start of Assistant B's Answer]
{answer_b}
[The End of Assistant B's Answer]"""
    
    completion = client.chat.completions.create(
        model='gpt-4o-2024-05-13',
        messages=[{'role': 'user', 'content': eval_prompt}],
        temperature=0.0
    )

    return completion

In [21]:
PROMPT_BASELINE = f"""以下の宿泊施設のレビューを5文以内で要約してください："""

In [22]:
reviews, _ = preprocess_reviews(path='./res/reviews.json')

def summarize(reviews, prompt, temperature=0.0, model='gpt-3.5-turbo-0125'):
    prompt = prompt + '\n\n' + reviews

    completion = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature
    )

    return completion

print(summarize(reviews, PROMPT_BASELINE).choices[0].message.content)

1. 和室湖畔側のお部屋で2家族10人で宿泊。夕食朝食ともに大満足。子供達は屋上スパに大満足。
2. 湖が見える良い部屋だったが、食事のタイミングが残念。天空スパは早朝に入り、ステキな体験。
3. 急な骨折で静養するために宿泊。スタッフの対応に感謝。眺めの良いお部屋で楽しい時間を過ごす。
4. 露天風呂が素晴らしかった。朝早く湖や山なみを見ながら入浴、幻想的な体験。
5. 冬の阿寒湖を楽しんだ。温泉、サウナ、料理全て満足。次は別の季節に再訪したい。


In [24]:
summary_gpt4 = summarize(reviews, PROMPT_BASELINE, model='gpt-4o-2024-05-13').choices[0].message.content

In [31]:
eval_count = 10

summaries_baseline = [summarize(reviews, PROMPT_BASELINE, temperature=1.0).choices[0].message.content for _ in range(eval_count)]
summaries_baseline

['1. 夕食と朝食に大満足、子供たちは屋上スパに大満足で、良い旅行になった。\n2. 屋上のスパが素晴らしく、バイキングの種類も豊富。ただ、残念な点もあった。\n3. スタッフの対応が素晴らしく、骨折のための要望にも柔軟に対応。湖の景色と施設に満足。\n4. 露天風呂が素晴らしく、朝の景色も幻想的で満足。料理の品数も豊富で期待通り。\n5. スタッフの対応が素晴らしく、料理やスパが充実していた。次回も利用したい。',
 '1. 和室湖畔側の清潔なお部屋で、夕食と朝食も大満足。子供達も屋上スパで大満足して楽しい旅行になった。\n2. 湖が見える部屋で18時のお食事は満席で急かされることが残念。バイキングは楽しく、早朝の天空スパの景色がステキだった。\n3. 急な骨折をしてしまったが、柔軟な対応で静養でき、スタッフの対応に感謝。眺めの良いお部屋で花火や湖を楽しめた。\n4. 美味しい食事と素晴らしい露天風呂。朝早い時間に入り、湖や山の幻想的な景色を楽しめた。\n5. 料理の充実、アップグレードしていただき感謝。屋上の天空スパやスタッフの対応に大満足。',
 '1. 和室湖畔側のお部屋で10人の2家族が大満足。子供達は屋上スパに大満足。\n2. 夕食や朝食は満腹満足。天空スパは朝早く入ると幻想的。\n3. スタッフの対応に感謝。食事やお風呂も満足。\n4. 露天風呂は最高。湖や山なみが幻想的。\n5. 料理の品数豊富で全て美味しい。露天風呂や食事で大満足。',
 '1. 2家族10人で利用。和室湖畔側のお部屋は清潔で景色もよく、夕食朝食も大満足。子供達は屋上スパに大満足し、楽しい旅行になった。\n2. 湖が見える良い部屋だったが、食事のタイミングが残念。バイキングの種類が豊富で、天空スパがステキ。\n3. 急なお話にも対応し、部屋や食事にも助かる配慮があり、良い旅行になった。スタッフの対応など、感謝の気持ちがある。\n4. 露天風呂が最高で、湖や山などの幻想的な景色を楽しんだ。食事も美味しいと満足。\n5. 冬の阿寒湖を楽しんだ宿泊。料理の品数、温泉、サービス全てに満足。露天風呂からの景色も最高。',
 '1. 和室湖畔側の清潔な部屋で大満足のファミリー旅行。子供たちは屋上スパに大満足。\n2. 夜8時の花火が部屋から見れなかったことが残念だったが、朝は天空スパで楽しい

In [ ]:
from tqdm import tqdm

def pairwise_eval_batch(reviews, answers_a, answers_b):
    a_cnt, b_cnt, draw_cnt = 0, 0, 0
    for i in tqdm(range(len(answers_a))):
        completion = pairwise_eval(reviews, answers_a[i], answers_b[i])
        verdict_text = completion.choices[0].message.content

        if '[[A]]' in verdict_text:
            a_cnt += 1
        elif '[[B]]' in verdict_text:
            b_cnt += 1
        elif '[[C]]' in verdict_text:
            draw_cnt += 1
        else:
            print('Evaluation Error')

    return a_cnt, b_cnt, draw_cnt

# gpt-3.5 + 단순 프롬프트 vs gpt-4o + 단순 프롬프트 (기존 비교)
print("=== gpt-3.5 (BASELINE) vs gpt-4o (BASELINE) ===")
wins, losses, ties = pairwise_eval_batch(reviews, summaries_baseline, [summary_gpt4 for _ in range(len(summaries_baseline))])
print(f'gpt-3.5 Wins: {wins}, gpt-4o Wins: {losses}, Ties: {ties}')

In [ ]:
# 핵심 비교: gpt-3.5 + 고도화 프롬프트 vs gpt-4o + 단순 프롬프트
print("=== gpt-3.5 (prompt_improved) vs gpt-4o (BASELINE) ===")
summaries_gpt35_improved = [summarize(reviews, prompt_improved, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries_gpt35_improved, [summary_gpt4 for _ in range(len(summaries_gpt35_improved))])
print(f'gpt-3.5+improved Wins: {wins}, gpt-4o+baseline Wins: {losses}, Ties: {ties}')

In [25]:
prompt = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。

要約結果は以下の条件を満たす必要があります：
1. すべての文は必ず丁寧語（です・ます調）で終わること。
2. 宿泊施設を紹介するトーン＆マナーで作成してください。
  2-1. 良い例
    a) 全体的に良い宿泊施設で、防音も問題なかったという評価です。
    b) 再訪予定だという声が多く見られます。
  2-2. 悪い例
    a) 良い宿泊施設で、防音も問題ありませんでした。
    b) 再訪予定です。
3. 要約結果は最低2文、最大5文で作成してください。
    
以下の宿泊施設のレビューを要約してください："""

eval_count = 10
summaries = [summarize(reviews, prompt, temperature=1.0).choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_gpt4 for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [00:45<00:00,  4.55s/it]

Wins: 0, Losses: 10, Ties: 0


In [26]:
import datetime
from dateutil import parser

def preprocess_reviews(path='./res/reviews.json'):
    with open(path, 'r', encoding='utf-8') as f:
        review_list = json.load(f)

    reviews_good, reviews_bad = [], []

    current_date = datetime.datetime.now()
    date_boundary = current_date - datetime.timedelta(days=6*30)

    filtered_cnt = 0
    for r in review_list:
        review_date_str = r['date']
        try:
            review_date = parser.parse(review_date_str)
        except (ValueError, TypeError):
            review_date = current_date

        if review_date < date_boundary:
            continue
        if len(r['review']) < 30:
            filtered_cnt += 1
            continue

        if r['star'] == 5:
            reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
        else:
            reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

    reviews_good = reviews_good[:min(len(reviews_good), 30)]
    reviews_bad = reviews_bad[:min(len(reviews_bad), 30)]

    reviews_good_text = '\n'.join(reviews_good)
    reviews_bad_text = '\n'.join(reviews_bad)

    return reviews_good_text, reviews_bad_text

reviews, _ = preprocess_reviews()

In [27]:
eval_count = 10
summaries = [summarize(reviews, prompt, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_gpt4 for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [01:04<00:00,  6.43s/it]

Wins: 0, Losses: 10, Ties: 0


In [ ]:
reviews_1shot, _ = preprocess_reviews(path='./res/reviews.json')
summary_1shot = summarize(reviews_1shot, prompt, temperature=0.0, model='gpt-4-turbo-2024-04-09').choices[0].message.content
prompt_1shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。

要約結果は以下の条件を満たす必要があります：
1. すべての文は必ず丁寧語（です・ます調）で終わること。
2. 宿泊施設を紹介するトーン＆マナーで作成してください。
  2-1. 良い例
    a) 全体的に良い宿泊施設で、防音も問題なかったという評価です。
    b) 再訪予定だという声が多く見られます。
  2-2. 悪い例
    a) 良い宿泊施設で、防音も問題ありませんでした。
    b) 再訪予定です。
3. 要約結果は最低2文、最大5文で作成してください。

以下はレビューと要約の例です。
例のレビュー：
{reviews_1shot}
例の要約結果：
{summary_1shot}
    
以下の宿泊施設のレビューを要約してください："""

summaries = [summarize(reviews, prompt, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_gpt4 for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [00:59<00:00,  5.93s/it]

Wins: 7, Losses: 3, Ties: 0


In [17]:
prompt_1shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。以下はレビューと要約の例です。
例のレビュー：
{reviews_1shot}
例の要約結果：
{summary_1shot}
    
以下の宿泊施設のレビューを要約してください："""

summaries = [summarize(reviews, prompt_1shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_gpt4 for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [00:58<00:00,  5.81s/it]

Wins: 4, Losses: 6, Ties: 0


In [18]:
reviews_1shot, _ = preprocess_reviews('./res/kunosato.json')
summary_1shot = summarize(reviews_1shot, prompt, temperature=0.0, model='gpt-4-turbo-2024-04-09').choices[0].message.content

In [19]:
reviews_2shot, _ = preprocess_reviews('./res/global_view.json')
summary_2shot = summarize(reviews_2shot, prompt_1shot, temperature=0.0, model='gpt-4-turbo-2024-04-09').choices[0].message.content

prompt_2shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。以下はレビューと要約の例です。

例のレビュー 1：
{reviews_1shot}
例の要約結果 1：
{summary_1shot}

例のレビュー 2：
{reviews_2shot}
例の要約結果 2：
{summary_2shot}
    
以下の宿泊施設のレビューを要約してください："""

summaries = [summarize(reviews, prompt_2shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_gpt4 for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [00:54<00:00,  5.46s/it]

Wins: 3, Losses: 7, Ties: 0
